# P128 — NeRF: representar escenas como campos de radiancia neuronal

## 1. Título y paper

**Paper:** *NeRF: Representing Scenes as Neural Radiance Fields for View Synthesis*  
**Autoría:** Ben Mildenhall, Pratul P. Srinivasan, Matthew Tancik, Jonathan T. Barron, Ravi Ramamoorthi, Ren Ng  
**Año y venue:** 2020 · ECCV 2020, 405–421  
**Nivel:** L3 · **Motor:** `nerf`  
**Ficha completa:** [`P128_nerf`](../../papers/foundational/P128_nerf/README.md)

**Hito:** Sustituye la escena explícita por una función continua que un perceptrón representa, y sintetiza vistas nuevas con una fidelidad que no se había visto.

- [doi:10.1007/978-3-030-58452-8_24](https://doi.org/10.1007/978-3-030-58452-8_24)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Representar una escena 3D como rejilla de vóxeles cuesta O(n³) en memoria: la resolución se paga al cubo y las rejillas finas no caben. Y las mallas exigen reconstruir geometría explícita, que falla con pelo, humo o vidrio.
2. Ejecutar una implementación mínima de la propuesta: Codificar la escena como una función continua que va de posición y dirección de vista a color y densidad, representada por un perceptrón multicapa, y renderizar integrando esa función a lo largo de cada rayo con la ecuación de volumen.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P02


## 4. Intuición

Guardar una escena como rejilla cuesta al cubo de la resolución. Guardarla como una **función** cuesta lo que ocupe el perceptrón — y no depende de la resolución en absoluto.


## 5. Concepto mínimo

```text
Rejilla explícita : O(n³)      lado 1024 → 17 180 MB
Función continua : |θ|        477 188 parámetros → 1,91 MB

Renderizar = integrar a lo largo del rayo:
    C = Σ T_i · (1 − e^(−σ_i·δ)) · c_i        con T_i = Π (1 − α_j)
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('nerf', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuánto ocupa una rejilla de lado 1024?
2. ¿Y el perceptrón equivalente?
3. ¿Quién decide que lo de delante tapa lo de atrás?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('nerf', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('nerf', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

**17 180 MB** frente a **1,91 MB**, y el perceptrón no crece con la resolución. La oclusión sale sola de la integral: la superficie a t=3 aporta **0,918** al color y lo que hay justo detrás, **0,075** — **12×** menos. Nadie programó esa regla.


## 10. Comentario pedagógico

La tercera cifra es la que se suele pasar por alto: la **codificación posicional**. Dos puntos a 0,005 de distancia están a 0,005 sin codificar y a **3,124** codificados, 625× más separados. Sin ella un perceptrón solo representa variaciones suaves, y una escena tiene bordes. NeRF sin codificación posicional produce una mancha borrosa.


## 11. Error o anti-patrón deliberado

Anti-patrón: leer «cabe en 2 MB» como «es barato».


In [ ]:
print('La memoria es minima. El coste esta en RENDERIZAR.')
print('Cada pixel exige decenas de consultas al perceptron a lo largo de su rayo.')
print('Por eso NeRF tardaba segundos por fotograma, y de ahi sale P132.')

## 12. Corrección

Los tres resultados, medidos:


In [ ]:
r = run_paper_lab('nerf', seed=3)['result']
for f in r['rejilla_explicita']:
    print(f)
print('mlp:', r['mlp'])
print('rayo:', r['composicion_del_rayo'])
print('codificacion:', r['codificacion_posicional'])

## 13. Desafío guiado

Explica por qué la composición por transmitancia hace innecesario decidir explícitamente qué superficie es visible, y qué ventaja da eso con humo o vidrio.


In [ ]:
r = run_paper_lab('nerf', seed=3)['result']
show(r)

## 14. Desafío autónomo

Calcula cuánta memoria necesitaría una rejilla que resolviera detalles de 1 mm en una habitación de 5 m. Compáralo con el perceptrón.


## 15. Evidencia de aprendizaje

Guarda el cálculo y la conclusión sobre cuál de las dos representaciones es viable.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P128_nerf/README.md) · evaluación formal: [`assessments/papers/P128_nerf.md`](../../assessments/papers/P128_nerf.md)


## 16. Cierre

La representación implícita gana en memoria y pierde en velocidad. Esa deuda se paga en P132.


## 17. Conexión con el siguiente hito

- P132

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
